# 05 · Decisão de modelo e roteamento (fronteira × simples)

**Pergunta:** qual é o modelo **mais barato** que ainda entrega a qualidade mínima aceitável — **por nível de complexidade**?

### Metodologia

1. **Teto de qualidade (baseline):** modelo de fronteira (`gpt-4.1`) + prompt bem construído (`v1`), no dataset congelado completo.
2. **Candidatos mais baratos** (`gpt-4.1-mini`, `gpt-4.1-nano`) no **mesmo dataset**, com resultados quebrados por complexidade.
3. **Escolha por nível:** o mais barato que atinge a qualidade mínima (definida a partir do baseline) **na nota do juiz e na tool selection**.
4. **Validação:** rodar o agente com o **roteador** ativo e comparar custo/qualidade contra o baseline.

O juiz é **sempre o mesmo** (`config.MODELO_JUIZ`) — mudar a régua no meio invalidaria a comparação.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # adiciona a raiz do projeto ao path

from receitas import config

## Critérios de decisão (definidos ANTES de ver os resultados)

Definir a régua antes evita escolher o critério que favorece o resultado que já se queria.

In [ ]:
# Qualidade mínima aceitável = 90% da nota do juiz que o modelo de fronteira atinge NAQUELE nível
FRACAO_DO_TETO = 0.90
# Tool selection: pode perder no máximo 0,10 de acerto exato em relação ao baseline
TOLERANCIA_TOOLS = 0.10
# Repetições por exemplo (3 = mede variância/consistência, mas custa 3x)
N_REPETICOES = 1

## Passo 1 e 2 — um experimento por modelo, mesmo dataset

In [ ]:
import pandas as pd
from langsmith import evaluate
from receitas.agente import criar_alvo
from receitas import avaliadores as av
from receitas.tendencia import resultados_para_df, registrar_no_historico

tabelas = []
for modelo in config.MODELOS_CANDIDATOS:  # fronteira primeiro = baseline
    resultados = evaluate(
        criar_alvo(modelo=modelo, versao_prompt="v1"),
        data=config.NOME_DATASET,
        evaluators=av.AVALIADORES_PADRAO,
        experiment_prefix=f"modelo-{modelo}",
        metadata={"modelo": modelo, "versao_prompt": "v1", "papel": "baseline" if modelo == config.MODELO_FRONTEIRA else "candidato"},
        num_repetitions=N_REPETICOES,
        max_concurrency=4,
    )
    tabela = resultados_para_df(resultados).assign(modelo_config=modelo)
    registrar_no_historico(tabela, rotulo=f"modelo/{modelo}")
    tabelas.append(tabela)
    print(f"{modelo}: {resultados.experiment_name}")

resultados_modelos = pd.concat(tabelas, ignore_index=True)

### Visão geral: qualidade × custo × latência por modelo

In [ ]:
METRICAS = {"juiz_qualidade": "mean", "tools_exatas": "mean", "schema_valido": "mean",
            "aproveitamento_prioritarios": "mean", "custo_usd": "mean", "latencia_s": "mean"}

geral = resultados_modelos.groupby("modelo_config").agg(METRICAS).reindex(config.MODELOS_CANDIDATOS)
geral["custo_por_ponto_qualidade"] = geral["custo_usd"] / geral["juiz_qualidade"]
geral.round(5)

### Quebrando por complexidade (o que realmente importa para o roteamento)

In [ ]:
NIVEIS = ["facil", "medio", "dificil"]
por_nivel = (resultados_modelos.groupby(["complexidade", "modelo_config"])
             .agg(METRICAS).reindex(pd.MultiIndex.from_product([NIVEIS, config.MODELOS_CANDIDATOS])))
por_nivel.round(4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from receitas.graficos import aplicar_estilo, COR_POR_MODELO, TINTA

aplicar_estilo()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8))

largura = 0.26
for i, modelo in enumerate(config.MODELOS_CANDIDATOS):
    valores = [por_nivel.loc[(n, modelo), "juiz_qualidade"] for n in NIVEIS]
    posicoes = np.arange(len(NIVEIS)) + (i - 1) * largura
    ax1.bar(posicoes, valores, width=largura - 0.03, color=COR_POR_MODELO[modelo], label=modelo)
    for x, v in zip(posicoes, valores):
        ax1.annotate(f"{v:.2f}", (x, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=7, color=TINTA)
ax1.set_xticks(range(len(NIVEIS)), labels=NIVEIS)
ax1.set_ylim(0, 1.1)
ax1.set_title("Nota do juiz por nível de complexidade", loc="left")
ax1.legend(loc="lower right")
ax1.grid(axis="x", visible=False)

for modelo in config.MODELOS_CANDIDATOS:
    linha = geral.loc[modelo]
    ax2.scatter(linha["custo_usd"], linha["juiz_qualidade"], s=90, color=COR_POR_MODELO[modelo],
                edgecolor="white", linewidth=2, zorder=3)
    ax2.annotate(modelo, (linha["custo_usd"], linha["juiz_qualidade"]), xytext=(8, -3),
                 textcoords="offset points", fontsize=8, color=TINTA)
ax2.set_xscale("log")
ax2.set_xlabel("custo médio por pedido (US$, log)")
ax2.set_ylabel("nota média do juiz")
ax2.set_title("Custo × qualidade (quanto mais acima e à esquerda, melhor)", loc="left")
plt.tight_layout()
plt.show()

### Variância (só quando `N_REPETICOES > 1`)

Desvio-padrão da nota **do mesmo exemplo** entre repetições, em média por modelo. Alto = modelo errático
(às vezes acerta, às vezes não) — um risco que a média sozinha esconde.

In [ ]:
if N_REPETICOES > 1:
    variancia = (resultados_modelos.groupby(["modelo_config", "complexidade", "id_cenario"])["juiz_qualidade"].std()
                 .groupby(["modelo_config", "complexidade"]).mean().unstack().reindex(config.MODELOS_CANDIDATOS))
    display(variancia.round(3))
else:
    print("N_REPETICOES = 1 -> variância não medida. Mude para 3 para ver a consistência de cada modelo.")

## Passo 3 — escolher o modelo por nível

Para cada nível, percorremos os modelos **do mais barato para o mais caro** e pegamos o primeiro que passa nos dois critérios.
Se nenhum candidato passa, fica o modelo de fronteira.

> Nos casos em que o juiz não se aplica (ex.: pergunta geral, sem receita), só a tool selection decide.

In [ ]:
mais_barato_primeiro = geral.sort_values("custo_usd").index.tolist()
decisoes, politica = [], {}

for nivel in NIVEIS:
    teto = por_nivel.loc[(nivel, config.MODELO_FRONTEIRA)]
    nota_minima = teto["juiz_qualidade"] * FRACAO_DO_TETO
    tools_minimo = teto["tools_exatas"] - TOLERANCIA_TOOLS

    escolhido = config.MODELO_FRONTEIRA
    for modelo in mais_barato_primeiro:
        linha = por_nivel.loc[(nivel, modelo)]
        passa_juiz = pd.isna(teto["juiz_qualidade"]) or linha["juiz_qualidade"] >= nota_minima
        passa_tools = linha["tools_exatas"] >= tools_minimo
        decisoes.append({"nível": nivel, "modelo": modelo, "nota juiz": round(linha["juiz_qualidade"], 3),
                         "mínimo juiz": round(nota_minima, 3), "tools exatas": round(linha["tools_exatas"], 3),
                         "mínimo tools": round(tools_minimo, 3), "custo médio US$": round(linha["custo_usd"], 5),
                         "passa?": "✅" if passa_juiz and passa_tools else "❌"})
        if passa_juiz and passa_tools:
            escolhido = modelo
            break
    politica[nivel] = escolhido

display(pd.DataFrame(decisoes))
print("Política de roteamento:", politica)

In [ ]:
import json
from datetime import datetime

config.ARQ_POLITICA.write_text(json.dumps({
    "politica": politica,
    "criterios": {"fracao_do_teto": FRACAO_DO_TETO, "tolerancia_tools": TOLERANCIA_TOOLS, "n_repeticoes": N_REPETICOES},
    "dataset": config.NOME_DATASET,
    "decidido_em": datetime.now().isoformat(timespec="seconds"),
}, indent=2, ensure_ascii=False))
print("Salvo em", config.ARQ_POLITICA)

## Passo 4 — validar o roteador ponta a ponta

Agora o modelo **não é fixo**: o roteador (`gpt-4.1-nano`, barato) classifica cada pedido e a política escolhe o modelo.
Medimos também se o roteador **acerta o nível** (`roteamento_correto`) — se ele erra para baixo, mandamos casos difíceis para um modelo fraco.
O custo do roteador entra na conta (ele também é uma chamada de LLM).

In [ ]:
resultados_roteado = evaluate(
    criar_alvo(politica=politica, versao_prompt="v1"),
    data=config.NOME_DATASET,
    evaluators=[*av.AVALIADORES_PADRAO, av.roteamento_correto],
    experiment_prefix="roteado",
    metadata={"modelo": "roteado", "politica": politica, "versao_prompt": "v1"},
    num_repetitions=N_REPETICOES,
    max_concurrency=4,
)
tabela_roteado = resultados_para_df(resultados_roteado).assign(modelo_config="roteado")
registrar_no_historico(tabela_roteado, rotulo="roteado")

baseline = resultados_modelos[resultados_modelos["modelo_config"] == config.MODELO_FRONTEIRA]
comparativo = pd.DataFrame({
    "baseline (gpt-4.1 sempre)": baseline[list(METRICAS)].mean(),
    "roteado": tabela_roteado[list(METRICAS)].mean(),
})
comparativo["variação"] = comparativo["roteado"] - comparativo["baseline (gpt-4.1 sempre)"]
display(comparativo.round(5))

economia = 1 - comparativo.loc["custo_usd", "roteado"] / comparativo.loc["custo_usd", "baseline (gpt-4.1 sempre)"]
print(f"Economia de custo: {economia:.0%}")
print(f"Roteador acertou o nível em {tabela_roteado['roteamento_correto'].mean():.0%} dos casos")
pd.crosstab(tabela_roteado["complexidade"], tabela_roteado["complexidade_roteada"], rownames=["nível real"], colnames=["roteado como"])

## Critério de decisão documentado

| Item | Regra |
|---|---|
| Teto de qualidade | nota do juiz do `gpt-4.1` + prompt `v1`, **por nível**, no dataset congelado `receitas-golden-v1` |
| Qualidade mínima aceitável | ≥ **90%** do teto na nota do juiz **e** tool selection exata no máximo **0,10** abaixo do baseline |
| Escolha | o modelo **mais barato** que cumpre os dois critérios naquele nível; senão, o de fronteira |
| Desempate / leitura de eficiência | **custo por ponto de qualidade** = custo médio ÷ nota média do juiz (menor é melhor) |
| Condição para manter o roteador | economia de custo relevante **e** queda de qualidade geral dentro do mínimo **e** roteador acertando o nível na grande maioria dos casos — errar para "fácil" num caso difícil é o erro caro |
| Quando reavaliar | ao mudar prompt, modelo, preço ou dataset: roda este notebook de novo; o histórico (`notebook 03`) mostra se a qualidade está melhorando ou piorando |

**Por que tool selection também decide?** Um modelo barato que chama tools a mais pode sair mais caro (mais voltas no loop do agente)
e um que esquece `verificar_validade` entrega receita bonita, mas ignora o que vai estragar — o juiz pode não punir o suficiente.

In [ ]:
print("Resumo para o registro de decisão:")
for nivel in NIVEIS:
    print(f"- {nivel:8s} -> {politica[nivel]}")
print(f"- economia vs. sempre-fronteira: {economia:.0%}")
print(f"- nota do juiz: baseline {comparativo.iloc[0, 0]:.3f} vs. roteado {comparativo.iloc[0, 1]:.3f}")